## Pupil supplement

In [ ]:
import numpy as np
import scipy.stats as stats
import scipy.io as sio
import pandas as pd

from dual_pfc_funcs import load_dict, getParams, jitter

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

# parameters
params = getParams()
subjects = params['subjects']
plot_sym = params['markers']
color_map = params['color_map']

In [ ]:
# pick example session for pupil trace
data_path = 'preprocessed_data/'
ex_sess,sub = 'Sa191206', 'satchel'
mat_fname = '{:s}/all_data_delay_{:s}.mat'.format(data_path, sub)
sub_dat = sio.loadmat(mat_fname,squeeze_me=True,struct_as_record=False)['all_data']
curr_dat = getattr(sub_dat,ex_sess)

pupil = getattr(curr_dat, 'pupil')
raw_vals = getattr(pupil, 'avg')
fast_vals = getattr(curr_dat, 'fast_component_pupil')
slow_vals = getattr(curr_dat, 'slow_component_pupil')

In [ ]:
fig,ax = plt.subplots(figsize=(6,5))
fig.supxlabel('trial number')
fig.supylabel('pupil diameter (a.u.)')

ax.plot(raw_vals,'darkgray')
ax.plot(slow_vals,'dimgray')

if SAVE_FIG:
    pdf = PdfPages('figs/raw_pupil_ex_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

lims = ax.get_ylim()

fig,ax = plt.subplots(2, 1,tight_layout=True,figsize=(6,5))
fig.supxlabel('trial number')
fig.supylabel('pupil diameter (a.u.)')

ax[0].plot(fast_vals,'k')
ax[1].plot(slow_vals,'dimgray')

ax[0].set_ylim(lims)
ax[1].set_ylim(lims)

if SAVE_FIG:
    pdf = PdfPages('figs/slowfast_pupil_ex_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# get evoked vals on high-mid-low latent trials for example session
example_sessions = [('Pe180717', 'pepe'),('Wa180211', 'wakko'),('Sa191206', 'satchel')]
pred_dat_all = load_dict(data_path + 'pupil_prediction_1d.pkl')

fig,ax = plt.subplots(2,3,tight_layout=True,figsize=(12,6))
xdata = np.arange(-200,400)
N = 600 # -200 to +400

for i_sub,(ex_sess, sub) in enumerate(example_sessions):

    # get pupil trial data
    mat_fname = '{:s}/all_data_delay_{:s}.mat'.format(data_path, sub)
    sub_dat = sio.loadmat(mat_fname,squeeze_me=True,struct_as_record=False)['all_data']
    curr_dat = getattr(sub_dat,ex_sess)

    pupil = getattr(curr_dat, 'pupil')
    raw_vals = getattr(pupil, 'avg')
    fast_vals = getattr(curr_dat, 'fast_component_pupil')
    slow_vals = getattr(curr_dat, 'slow_component_pupil')
    baseline_vals = getattr(pupil, 'baseline')

    pupil_traces = getattr(pupil, 'trial')
    baseline_pupil = np.zeros((len(pupil_traces),N))
    for i in range(len(pupil_traces)):
        baseline_pupil[i,:] = pupil_traces[i][:N] # - slow_vals[i]
    avg_trace = np.mean(baseline_pupil, axis=0)

    # plot average traces on percentile trials
    ax[0,i_sub].plot(xdata, avg_trace,color='k')

    ax[0,i_sub].set_title('{} example session'.format(sub))
    ax[0,i_sub].set_xlabel('time from target onset (ms)')
    ax[0,i_sub].set_ylabel('pupil diameter (a.u.)')

    # plot average evoked traces on percentile trials
    baseline_pupil = np.zeros((len(pupil_traces),N))
    for i in range(len(pupil_traces)):
        baseline_pupil[i,:] = pupil_traces[i][:N] - baseline_vals[i]
    avg_trace = np.mean(baseline_pupil, axis=0)

    ax[1,i_sub].plot(xdata, avg_trace,color='k')

    ax[1,i_sub].set_title('baseline subtraction')
    ax[1,i_sub].set_xlabel('time from target onset (ms)')
    ax[1,i_sub].set_ylabel('pupil diameter (a.u.)')

if SAVE_FIG:
    pdf = PdfPages('figs/evoked_pupil_avg_ex_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# load prediction data
CROSSVAL = False
data_path = 'preprocessed_data/'
if CROSSVAL:
    dat = load_dict(data_path + 'pupil_prediction_cv.pkl')
    evoked_dat = load_dict(data_path + 'evoked_pupil_prediction_cv.pkl')
else:
    dat = load_dict(data_path + 'pupil_prediction.pkl')
    evoked_dat = load_dict(data_path + 'evoked_pupil_prediction.pkl')
fnames = evoked_dat.keys()

In [ ]:
# corr between evoked and trial
df = pd.DataFrame(columns=['Subject', 'SessionName', 'PupilCorr'])
for j,(sub,sym) in enumerate(zip(subjects,plot_sym)):
    for sess in fnames:
        if sess[:2]==sub[:2].title():
            # trial-by-trial pupil
            trial_vals = dat[sess]['pupil_zsc'].squeeze()
            # evoked pupil
            evoked_vals = evoked_dat[sess]['pupil_zsc']
            # corr
            r = np.corrcoef(trial_vals, evoked_vals)[0,1]
            df2 = {'Subject':sub, 'SessionName':sess, 'PupilCorr':r}
            df.loc[len(df)] = df2

# plot box and whisker of session means
fig,ax = plt.subplots()

for j,(sub,sym) in enumerate(zip(subjects,plot_sym)):
    pupil_corrs = df[df['Subject']==sub]['PupilCorr'].to_numpy()
    ydata = (2-j) + 0.3 + jitter(len(pupil_corrs),spacing=0.1)
    ax.scatter(pupil_corrs, ydata, color='k',alpha=0.5,s=12,marker=sym,edgecolors='none')
    ax.barh((2-j), np.mean(pupil_corrs), color='k', xerr=stats.sem(pupil_corrs), align='center', ecolor='black', edgecolor='black', height=0.3)
ax.axvline(linewidth=1, color='k', linestyle='--')

# formatting
ax.set_xlabel('correlation between fast and event-related pupil')
ax.set_xlim([-1,1])
plt.yticks([2,1,0], ['Monkey {:s}'.format(sub[:2].title()) for sub in subjects])

if SAVE_FIG:
    pdf = PdfPages('figs/pupil_corr_all_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# statistics 
alpha = 0.01 
print("correlation statistics across sessions:")
p = stats.ttest_1samp(df['PupilCorr'], popmean=0,alternative='less').pvalue
print("  correlation < 0? {}, p = {:.6f}".format(p < alpha, p))

print()
for sub in subjects:
    filt = df['Subject'] == sub
    tmp_df = df[filt]
    print('Subject: {}, mean correlation = {:.3f}'.format(sub, np.mean(tmp_df['PupilCorr'])))

    # test if across and within are different for this monkey
    p = stats.ttest_1samp(tmp_df['PupilCorr'], popmean=0,alternative='less').pvalue
    print("  correlation < 0? {}, p = {:.6f}".format(p < alpha, p))

In [ ]:
# evoked prediction
col = 'k'
fig,ax = plt.subplots()
for i,latent in enumerate(['across','within-right','within-left']):
    for j,sub in enumerate(subjects):
        # trial-by-trial pupil
        curr = np.array([evoked_dat[sess]['r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()])
        if i==0:
            ax.errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6,label=f'Monkey {sub[:2].title()}')
        else:
            ax.errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6)

        # evoked pupil
        null = np.array([evoked_dat[sess]['null_r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()]).flatten()
        null_prc = np.percentile(null,[2.5,97.5])
        ax.plot(i*3+j*.7,null.mean(),color='k',alpha=0.4,linewidth=3)
        ax.plot([i*3+j*.7,i*3+j*.7],null_prc,'k-',alpha=0.4,linewidth=3)
        
# formatting
ax.legend()
ax.set_xticks(ticks=[0.7,3.7,6.7])
ax.set_xticklabels(['across-area','within-area \n (right)','within-area \n (left)'])
ax.set_ylabel('event-related pupil \n diameter prediction ($r^2$)')

colors = [color_map['across'],color_map['within1'],color_map['within2']]
for xtick, color in zip(ax.get_xticklabels(), colors):
    xtick.set_color(color)

if SAVE_FIG:
    pdf = PdfPages('figs/evoked_pred_all_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# Statistics for evoked: ['evoked']
print('Evoked statistics')
all_across, all_left, all_right = np.array([]),np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([evoked_dat[sess]['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([evoked_dat[sess]['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([evoked_dat[sess]['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])

    # pool
    all_across = np.concatenate((all_across, across))
    all_left = np.concatenate((all_left, within_left))
    all_right = np.concatenate((all_right, within_right))

    p = stats.ttest_rel(a=across, b=within_left,alternative='greater').pvalue
    print(f'{sub} across > within-left, p={p:.6f}')
    p = stats.ttest_rel(a=across, b=within_right,alternative='greater').pvalue
    print(f'{sub} across > within-right, p={p:.6f}')

    diffs = np.concatenate([across - within_left, across - within_right]) # across - within
    p = stats.ttest_1samp(diffs, popmean=0, alternative='greater').pvalue
    print(f'{sub} pooled across > within, p={p:.6f}')

# pooled:
print()
print('Average r2 values: across {:.3f}, within left {:.3f}, within right {:.3f}'.format(all_across.mean(), all_left.mean(), all_right.mean()))
# pooled across areas
diffs = np.concatenate([all_across - all_left, all_across - all_right]) # across - within
p = stats.ttest_1samp(diffs, popmean=0, alternative='greater').pvalue
print(f'pooled across > within, p={p:.6f}')

print()

# repeat analysis with 1D predictions:
if CROSSVAL:
    control_dat = load_dict(data_path + 'evoked_pupil_prediction_1d_cv.pkl')
else:
    control_dat = load_dict(data_path + 'evoked_pupil_prediction_1d.pkl')
fnames = control_dat.keys()
print('Evoked control - one dimensional')

# Statistics for evoked: ['evoked']
all_across, all_left, all_right = np.array([]),np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([control_dat[sess]['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([control_dat[sess]['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([control_dat[sess]['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])
    # pool
    all_across = np.concatenate((all_across, across))
    all_left = np.concatenate((all_left, within_left))
    all_right = np.concatenate((all_right, within_right))

# pooled:
print('Average r2 values: across {:.3f}, within left {:.3f}, within right {:.3f}'.format(all_across.mean(), all_left.mean(), all_right.mean()))
# pooled across areas
diffs = np.concatenate([all_across - all_left, all_across - all_right]) # across - within
p = stats.ttest_1samp(diffs, popmean=0, alternative='greater').pvalue
print(f'pooled across > within, p={p:.6f}')

In [ ]:
# control: are fast and evoked different?
data_path = 'preprocessed_data/'
dat = load_dict(data_path + 'evoked_resid_pupil_prediction.pkl')
fnames = dat.keys()

col = 'k'
fig,ax = plt.subplots()
for i,latent in enumerate(['across','within-right','within-left']):
    for j,sub in enumerate(subjects):
        # trial-by-trial pupil
        curr = np.array([dat[sess]['r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()])
        if i==0:
            ax.errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6,label=f'Monkey {sub[:2].title()}')
        else:
            ax.errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6)

        # null dist
        null = np.array([dat[sess]['null_r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()]).flatten()
        null_prc = np.percentile(null,[2.5,97.5])
        ax.plot([i*3+j*.7,i*3+j*.7],null_prc,'k-',alpha=0.4,linewidth=3)
        
# formatting
ax.set_xticks(ticks=[0.7,3.7,6.7])
ax.set_xticklabels(['across-area','within-area \n (right)','within-area \n (left)'])
ax.legend()
ax.set_ylabel('residual event-related pupil diameter \n variance explained ($r^2$)')

colors = [color_map['across'],color_map['within1'],color_map['within2']]
for xtick, color in zip(ax.get_xticklabels(), colors):
    xtick.set_color(color)
fig.show()

# statistics
print('Event-related statistics')
all_across, all_left, all_right = np.array([]),np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([dat[sess]['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([dat[sess]['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([dat[sess]['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])
    
    # combine across areas
    all_across = np.concatenate((all_across, across))
    all_left = np.concatenate((all_left, within_left))
    all_right = np.concatenate((all_right, within_right))

# pooled:
print()
print('Average r2 values: across {:.3f}, within left {:.3f}, within right {:.3f}'.format(all_across.mean(), all_left.mean(), all_right.mean()))
# pooled across areas
diffs = np.concatenate([all_across - all_left, all_across - all_right]) # across - within
p = stats.ttest_1samp(diffs, popmean=0, alternative='greater').pvalue
print(f'pooled residual across > within, p={p:.6f}')

In [ ]:
# Statistics:
print()
print('Across to null statistics')
all_across, all_null = np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([dat[sess]['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    null = np.array([dat[sess]['null_r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()]).flatten()
    
    # combine across areas
    all_across = np.concatenate((all_across, across))
    all_null = np.concatenate((all_null, null))

    # per subject stat
    p = stats.ttest_1samp(a=across, popmean=np.mean(null), alternative='greater').pvalue
    print(f'  {sub} across > null, p={p:.6f}')

# pooled:
print()
print('Average r2 values: across {:.3f}'.format(all_across.mean()))
print('Pooled t test')
p = stats.ttest_1samp(a=all_across, popmean=np.mean(all_null), alternative='greater').pvalue
print(f'  across > null, p={p:.6f}')